#Imporve pronounciation

**pipeline:**

YouTube Noorani Qaida → download once → approved timestamps → FFmpeg → 28 local reference clips + metadata → child selects letter → play reference → child records → Wav2Vec2 → target-letter score / 100 → feedback.


In [2]:
!pip -q install -U transformers librosa soundfile gradio
!apt -qq update > /dev/null && apt -qq install -y ffmpeg > /dev/null



W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)




In [ ]:
import os
import subprocess
import numpy as np
import pandas as pd
import librosa
import torch
import gradio as gr

from transformers import Wav2Vec2FeatureExtractor, Wav2Vec2ForSequenceClassification
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TARGET_SR = 16000

# Dataset paths
DATASET_DIR = "..\\datasets"

SOURCE_AUDIO = os.path.join(DATASET_DIR, "Audio.mp4")

REFERENCE_DIR = os.path.join(DATASET_DIR, "reference_audio")
METADATA_FILE = os.path.join(DATASET_DIR, "metadata.csv")

os.makedirs(REFERENCE_DIR, exist_ok=True)

if not os.path.exists(SOURCE_AUDIO):
    raise FileNotFoundError(
        f"Audio.mp4 not found:\n{SOURCE_AUDIO}"
    )

print("Source file:", SOURCE_AUDIO)

Source file: datasets/Audio.mp4


In [6]:
letter_clips = {
    "ا": (34, 36), "ب": (36, 38), "ت": (38, 40), "ث": (40, 43),
    "ج": (43, 45), "ح": (45, 48), "خ": (48, 50), "د": (50, 53),
    "ذ": (53, 55), "ر": (55, 58), "ز": (58, 60), "س": (60, 63),
    "ش": (63, 66), "ص": (66, 69), "ض": (70, 72), "ط": (72, 75),
    "ظ": (75, 77), "ع": (77, 79), "غ": (80, 82), "ف": (83, 85),
    "ق": (85, 88), "ك": (88, 90), "ل": (90, 93), "م": (93, 96),
    "ن": (96, 98), "و": (98, 100), "ه": (101, 103), "ي": (105, 108)
}

assert len(letter_clips) == 28

print(f"Number of letters: {len(letter_clips)}")

Number of letters: 28


In [7]:
def build_reference_dataset():

    rows = []

    for letter, (start, end) in letter_clips.items():
        output_file = os.path.join(REFERENCE_DIR,  f"{letter}.wav")

        if not os.path.exists(output_file):
            print(f"Creating {letter}: {start}s → {end}s")
            command = [
                "ffmpeg",
                "-y",
                "-i", SOURCE_AUDIO,
                "-ss", str(start),
                "-to", str(end),
                "-vn",
                "-ac", "1",
                "-ar", str(TARGET_SR),
                "-c:a", "pcm_s16le",
                output_file
            ]

            subprocess.run(
                command,
                check=True,
                stdout=subprocess.DEVNULL,
                stderr=subprocess.DEVNULL
            )

        rows.append({
            "letter": letter,
            "audio_file": f"reference_audio/{letter}.wav",
            "start_seconds": start,
            "end_seconds": end
        })

    metadata = pd.DataFrame(rows)

    metadata.to_csv(
        METADATA_FILE,
        index=False,
        encoding="utf-8-sig"
    )

    print("\nReference dataset created")
    print(f"Clips: {len(metadata)}")
    print(f"Metadata: {METADATA_FILE}")

    return metadata

metadata = build_reference_dataset()
display(metadata)

Creating ا: 34s → 36s
Creating ب: 36s → 38s
Creating ت: 38s → 40s
Creating ث: 40s → 43s
Creating ج: 43s → 45s
Creating ح: 45s → 48s
Creating خ: 48s → 50s
Creating د: 50s → 53s
Creating ذ: 53s → 55s
Creating ر: 55s → 58s
Creating ز: 58s → 60s
Creating س: 60s → 63s
Creating ش: 63s → 66s
Creating ص: 66s → 69s
Creating ض: 70s → 72s
Creating ط: 72s → 75s
Creating ظ: 75s → 77s
Creating ع: 77s → 79s
Creating غ: 80s → 82s
Creating ف: 83s → 85s
Creating ق: 85s → 88s
Creating ك: 88s → 90s
Creating ل: 90s → 93s
Creating م: 93s → 96s
Creating ن: 96s → 98s
Creating و: 98s → 100s
Creating ه: 101s → 103s
Creating ي: 105s → 108s

Reference dataset created
Clips: 28
Metadata: datasets/metadata.csv


,letter,audio_file,start_seconds,end_seconds
0,ا,reference_audio/ا.wav,34,36
1,ب,reference_audio/ب.wav,36,38
2,ت,reference_audio/ت.wav,38,40
3,ث,reference_audio/ث.wav,40,43
4,ج,reference_audio/ج.wav,43,45
5,ح,reference_audio/ح.wav,45,48
6,خ,reference_audio/خ.wav,48,50
7,د,reference_audio/د.wav,50,53
8,ذ,reference_audio/ذ.wav,53,55
9,ر,reference_audio/ر.wav,55,58


In [8]:
metadata = pd.read_csv(METADATA_FILE, encoding="utf-8-sig")
arabic_letters = metadata["letter"].tolist()

print("Arabic letters:")
print(arabic_letters)
print("Number of letters:", len(arabic_letters))

Arabic letters:
['ا', 'ب', 'ت', 'ث', 'ج', 'ح', 'خ', 'د', 'ذ', 'ر', 'ز', 'س', 'ش', 'ص', 'ض', 'ط', 'ظ', 'ع', 'غ', 'ف', 'ق', 'ك', 'ل', 'م', 'ن', 'و', 'ه', 'ي']
Number of letters: 28


In [9]:
def get_reference_audio(letter):

    if letter not in arabic_letters:
        raise ValueError(f"Unknown Arabic letter: {letter}")

    row = metadata[metadata["letter"] == letter].iloc[0]
    path = os.path.join(DATASET_DIR, row["audio_file"])

    if not os.path.exists(path):
        raise FileNotFoundError(f"Reference audio not found:\n{path}")

    return path

In [10]:
print(get_reference_audio("ب"))

datasets/reference_audio/ب.wav


In [11]:
MODEL_ID = "masumtechnonext/wav2vec2-arabic-letter-verifier"

processor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_ID)
model = Wav2Vec2ForSequenceClassification.from_pretrained(MODEL_ID).to(device)
model.eval()

print("Model loaded")

preprocessor_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/3.36k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.26GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

Model loaded


In [12]:
label_to_letter = {
    "Ain": "ع",
    "Alif": "ا",
    "Ba": "ب",
    "Daad": "ض",
    "Dal": "د",
    "Dhaa": "ظ",
    "Dhal": "ذ",
    "Faa": "ف",
    "Ghain": "غ",
    "Ha": "ه",
    "Haa": "ح",
    "Jeem": "ج",
    "Kaf": "ك",
    "Kha": "خ",
    "Laam": "ل",
    "Meem": "م",
    "Noon": "ن",
    "Qaf": "ق",
    "Raa": "ر",
    "Saad": "ص",
    "Seen": "س",
    "Sheen": "ش",
    "Ta": "ت",
    "Taa": "ط",
    "Tha": "ث",
    "Unknown": "Unknown",
    "Waw": "و",
    "Yaa": "ي",
    "Zay": "ز"
}

In [13]:
def preprocess_audio(audio, sample_rate=TARGET_SR):

    audio = np.asarray(audio, dtype=np.float32)

    # Convert stereo → mono
    if audio.ndim > 1:
        audio = np.mean(audio, axis=1)

    # Remove silence
    audio, _ = librosa.effects.trim(audio, top_db=25)

    if len(audio) == 0:
        raise ValueError("لم يتم اكتشاف صوت.")

    # Normalize
    peak = np.max(np.abs(audio))

    if peak > 0:
        audio = audio / peak

    # Resample
    if sample_rate != TARGET_SR:
        audio = librosa.resample(
            audio,
            orig_sr=sample_rate,
            target_sr=TARGET_SR
        )

    return audio

In [14]:
def predict_letter(audio, target_letter):
    audio = preprocess_audio(audio, TARGET_SR)
    inputs = processor(
        audio,
        sampling_rate=TARGET_SR,
        return_tensors="pt"
    )

    with torch.no_grad():
        logits = model(inputs.input_values.to(device)).logits

    probabilities = torch.softmax(logits, dim=-1)[0]

    # Predicted class
    pred_id = int(torch.argmax(probabilities))
    predicted_label = model.config.id2label[pred_id]

    predicted_letter = label_to_letter.get(predicted_label, "Unknown")
    confidence = (float(probabilities[pred_id]) * 100)

    # Target class
    target_label = next(
        (
            label
            for label, letter in label_to_letter.items()
            if letter == target_letter
        ),
        None
    )

    target_id = next(
        (
            int(idx)
            for idx, label in model.config.id2label.items()
            if label == target_label
        ),
        None
    )

    target_probability = (
        float(probabilities[target_id]) * 100
        if target_id is not None
        else 0.0
    )

    is_correct = predicted_letter == target_letter
    score = round(target_probability, 2)

    # Feedback
    if is_correct:
        if score >= 90:
            feedback = "ممتاز!"
        elif score >= 75:
            feedback = "أحسنت!"
        elif score >= 60:
            feedback = "جيد! حاول مرة أخرى لدرجة أعلى."
        else:
            feedback = "قريب! اسمع المثال وحاول مرة أخرى."

    elif predicted_letter == "Unknown":
        feedback = (
            f"لم أتعرف على الحرف {target_letter}. "
            "اسمع المثال وحاول مرة أخرى."
        )

    else:
        feedback = (
            f"النموذج سمع {predicted_letter}. "
            f"المطلوب {target_letter}. "
            "اسمع المثال وحاول مرة أخرى."
        )

    return {
        "target_letter": target_letter,
        "predicted_letter": predicted_letter,
        "predicted_label": predicted_label,
        "confidence": round(confidence, 2),
        "target_probability": round(target_probability, 2),
        "score": score,
        "is_correct": is_correct,
        "feedback": feedback
    }

In [15]:
class PronunciationEvaluator:

    def __init__(self, threshold=86.0):
        self.threshold = threshold

    def evaluate(self, audio, target_letter):
        result = predict_letter(audio, target_letter)
        accepted = (
            result["predicted_letter"] == target_letter
            and result["target_probability"] >= self.threshold
        )

        if accepted:
            return {
                "success": True,
                "score": result["score"],
                "message": (
                    f"أتقنت الحرف بنسبة "
                    f"{result['score']:.0f}%"
                )
            }

        return {
            "success": False,
            "score": 0,
            "message": "أعد التسجيل وحاول مرة ثانية."
        }

pronunciation_evaluator = PronunciationEvaluator(threshold=75.0)

In [16]:
def analyze_child_voice(audio_path, target_letter):

    if audio_path is None:
        return ("سجل صوتك أولًا", 0, "حاول تسجيل الحرف مرة أخرى.")

    try:
        audio, sr = librosa.load(
            audio_path,
            sr=TARGET_SR,
            mono=True
        )

        result = pronunciation_evaluator.evaluate(audio, target_letter)

        if result["success"]:
            return  "أحسنت! نطقت الحرف بشكل صحيح", result["score"], result["message"]

        return "أعد التسجيل", 0, result["message"]

    except Exception as e:
        return "حدث خطأ أثناء تحليل الصوت", 0, str(e)

In [17]:
def show_letter(letter):
    reference_path = get_reference_audio(letter)
    return (f"الحرف المختار: {letter}", reference_path)

with gr.Blocks(title="تعلم الحروف العربية") as demo:
    gr.Markdown(
        """
        # تعلم الحروف العربية
        اختر حرفًا → اسمع → قلد → احصل على تقييمك
        """
    )

    with gr.Row():
        with gr.Column():
            letter = gr.Dropdown(
                choices=arabic_letters,
                value="ا",
                label="اختر الحرف"
            )

            letter_display = gr.Textbox(
                value="الحرف المختار: ا",
                label="الحرف",
                interactive=False
            )

            reference_audio = gr.Audio(
                value=get_reference_audio("ا"),
                type="filepath",
                label="النطق الصحيح",
                interactive=False
            )

            child_audio = gr.Audio(
                sources=["microphone"],
                type="filepath",
                format="wav",
                label="سجل نطقك"
            )

            analyze_button = gr.Button(
                "حلّل نطقي",
                variant="primary"
            )

    gr.Markdown("---")
    gr.Markdown("## تقييم النطق")
    with gr.Row():
        result_status = gr.Textbox(label="النتيجة")
        score_output = gr.Number(label="Score / 100")

    feedback_output = gr.Textbox(label="Feedback")

    letter.change(
        fn=show_letter,
        inputs=letter,
        outputs=[letter_display, reference_audio]
    )

    analyze_button.click(
        fn=analyze_child_voice,
        inputs=[child_audio, letter],
        outputs=[result_status, score_output, feedback_output]
    )


demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2c2bd34213298437df.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
